In [0]:
display(dbutils.fs.ls("/Volumes/workspace/bronze/raw_files/"))


path,name,size,modificationTime
dbfs:/Volumes/workspace/bronze/raw_files/bikeshare_2025_01.csv,bikeshare_2025_01.csv,27877999,1791493496000
dbfs:/Volumes/workspace/bronze/raw_files/bikeshare_2025_02.csv,bikeshare_2025_02.csv,17270549,1791493494000
dbfs:/Volumes/workspace/bronze/raw_files/bikeshare_2025_03.csv,bikeshare_2025_03.csv,42890894,1791493499000
dbfs:/Volumes/workspace/bronze/raw_files/bikeshare_2025_04.csv,bikeshare_2025_04.csv,64769987,1791493503000
dbfs:/Volumes/workspace/bronze/raw_files/bikeshare_2025_05.csv,bikeshare_2025_05.csv,96023924,1791493507000
dbfs:/Volumes/workspace/bronze/raw_files/bikeshare_2025_06.csv,bikeshare_2025_06.csv,134578131,1791493508000
dbfs:/Volumes/workspace/bronze/raw_files/bikeshare_2025_07.csv,bikeshare_2025_07.csv,163449207,1791493508000
dbfs:/Volumes/workspace/bronze/raw_files/bikeshare_2025_08.csv,bikeshare_2025_08.csv,159702596,1791493508000
dbfs:/Volumes/workspace/bronze/raw_files/bikeshare_2025_09.csv,bikeshare_2025_09.csv,145556211,1791493508000
dbfs:/Volumes/workspace/bronze/raw_files/bikeshare_2025_10.csv,bikeshare_2025_10.csv,119252911,1791493508000


In [0]:
from pyspark.sql import functions as F

trips_df = (spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("/Volumes/workspace/bronze/raw_files/bikeshare_2025_*.csv"))

# standardize column names to snake_case
for c in trips_df.columns:
    trips_df = trips_df.withColumnRenamed(c, c.strip().lower().replace(" ", "_"))

trips_df = (trips_df
    .withColumn("_load_ts", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path")))

trips_df.write.mode("overwrite").saveAsTable("workspace.bronze.trips_raw")

In [0]:
display(spark.sql("SELECT _source_file, COUNT(*) AS row_count FROM workspace.bronze.trips_raw GROUP BY 1 ORDER BY 1"))

_source_file,row_count
dbfs:/Volumes/workspace/bronze/raw_files/bikeshare_2025_01.csv,202946
dbfs:/Volumes/workspace/bronze/raw_files/bikeshare_2025_02.csv,125670
dbfs:/Volumes/workspace/bronze/raw_files/bikeshare_2025_03.csv,312178
dbfs:/Volumes/workspace/bronze/raw_files/bikeshare_2025_04.csv,471525
dbfs:/Volumes/workspace/bronze/raw_files/bikeshare_2025_05.csv,699461
dbfs:/Volumes/workspace/bronze/raw_files/bikeshare_2025_06.csv,980624
dbfs:/Volumes/workspace/bronze/raw_files/bikeshare_2025_07.csv,1192062
dbfs:/Volumes/workspace/bronze/raw_files/bikeshare_2025_08.csv,1164697
dbfs:/Volumes/workspace/bronze/raw_files/bikeshare_2025_09.csv,1062001
dbfs:/Volumes/workspace/bronze/raw_files/bikeshare_2025_10.csv,872482


In [0]:
import pandas as pd
from pyspark.sql import functions as F

# pandas skips the first 2 metadata rows, reads real headers from row 3
pdf = pd.read_csv("/Volumes/workspace/bronze/raw_files/toronto_weather_2025.csv", skiprows=2)

weather_df = spark.createDataFrame(pdf)

# clean column names: lowercase, strip units like "(°C)" and "(mm)"
import re
for c in weather_df.columns:
    clean = re.sub(r"\s*\(.*?\)", "", c).strip().lower().replace(" ", "_")
    weather_df = weather_df.withColumnRenamed(c, clean)

weather_df = weather_df.withColumn("_load_ts", F.current_timestamp())

weather_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.bronze.weather_raw")

In [0]:
display(spark.sql("SELECT * FROM workspace.bronze.weather_raw LIMIT 5"))

time,temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum,rain_sum,windspeed_10m_max,_load_ts
2025-01-01,3.7,0.2,1.9,2.0,2.0,23.0,2026-10-09T19:26:10.680Z
2025-01-02,0.5,-2.8,-0.9,0.0,0.0,25.0,2026-10-09T19:26:10.680Z
2025-01-03,-0.5,-5.9,-3.0,0.1,0.0,24.0,2026-10-09T19:26:10.680Z
2025-01-04,-2.4,-5.5,-4.2,0.8,0.0,31.9,2026-10-09T19:26:10.680Z
2025-01-05,-2.2,-4.8,-3.9,0.4,0.0,23.8,2026-10-09T19:26:10.680Z


In [0]:
display(spark.sql("SELECT COUNT(*) FROM workspace.bronze.weather_raw"))

COUNT(*)
365


In [0]:
display(spark.sql("SELECT _source_file, COUNT(*) AS row_count FROM workspace.bronze.trips_raw GROUP BY 1 ORDER BY 1"))

_source_file,row_count
dbfs:/Volumes/workspace/bronze/raw_files/bikeshare_2025_01.csv,202946
dbfs:/Volumes/workspace/bronze/raw_files/bikeshare_2025_02.csv,125670
dbfs:/Volumes/workspace/bronze/raw_files/bikeshare_2025_03.csv,312178
dbfs:/Volumes/workspace/bronze/raw_files/bikeshare_2025_04.csv,471525
dbfs:/Volumes/workspace/bronze/raw_files/bikeshare_2025_05.csv,699461
dbfs:/Volumes/workspace/bronze/raw_files/bikeshare_2025_06.csv,980624
dbfs:/Volumes/workspace/bronze/raw_files/bikeshare_2025_07.csv,1192062
dbfs:/Volumes/workspace/bronze/raw_files/bikeshare_2025_08.csv,1164697
dbfs:/Volumes/workspace/bronze/raw_files/bikeshare_2025_09.csv,1062001
dbfs:/Volumes/workspace/bronze/raw_files/bikeshare_2025_10.csv,872482


In [0]:
# check for nulls in key columns
display(spark.sql("""
    SELECT 
        SUM(CASE WHEN start_station_name IS NULL THEN 1 ELSE 0 END) AS null_start_station,
        SUM(CASE WHEN end_station_name IS NULL THEN 1 ELSE 0 END) AS null_end_station,
        SUM(CASE WHEN trip_duration IS NULL THEN 1 ELSE 0 END) AS null_duration
    FROM workspace.bronze.trips_raw
"""))

null_start_station,null_end_station,null_duration
7,924,0


In [0]:
display(spark.sql("""
    SELECT MIN(trip_duration) AS min_dur, MAX(trip_duration) AS max_dur, AVG(trip_duration) AS avg_dur
    FROM workspace.bronze.trips_raw
"""))

min_dur,max_dur,avg_dur
0,151037,972.2978548534916
